# Chess ML - Model Training V2

## Improvements over V1
| | V1 | V2 |
|---|---|---|
| **Board encoding** | 65 integers (-6 to +6) | 12-plane binary (781 features) |
| **Training data** | 10K positions | 148K positions (all available) |
| **Architecture** | 3 hidden layers, no BN | 3 hidden layers + BatchNorm |
| **Training** | Fixed 20 epochs | Early stopping (best weights restored) |
| **Evaluation** | Top-1 only | Top-1, Top-3, Top-5 |
| **Inference** | Raw softmax | Legal move masking |

### Why 12-plane encoding?
Rather than encoding each square as a single integer (-6=black king, +6=white king),
we use one binary plane per piece type per color (6 piece types × 2 colors = 12 planes, each 64 squares).
This preserves spatial structure and is the same representation used by AlphaZero and Leela Chess Zero.

In [2]:
import pandas as pd
import numpy as np
import os
import pickle
import chess
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('Libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

Libraries imported successfully!
TensorFlow version: 2.12.0
GPU available: []


In [3]:
DB_USER = os.getenv('USER')
DB_NAME = 'chess_app'
DATABASE_URL = f'postgresql://{DB_USER}:@localhost:5432/{DB_NAME}'

engine = create_engine(DATABASE_URL, echo=False)

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM positions;')).fetchone()[0]
    print(f'Connected to database!')
    print(f'Total positions available: {count:,}')

Connected to database!
Total positions available: 148,320


## Step 1: Load All Positions
V1 used only 10K of 148K available positions. We use everything now.

In [5]:
print('Loading all positions from database...')

positions_df = pd.read_sql("""
    SELECT 
        p.fen,
        p.move_played,
        g.white_elo,
        g.black_elo
    FROM positions p
    JOIN games g ON p.game_id = g.game_id
    WHERE g.white_elo > 1400
    ORDER BY RANDOM();
""", engine)

print(f'Loaded {len(positions_df):,} positions')
print(f'Unique moves: {positions_df["move_played"].nunique():,}')
print(f'\nSample:')
positions_df.head()

Loading all positions from database...
Loaded 112,466 positions
Unique moves: 1,841

Sample:


,fen,move_played,white_elo,black_elo
0,rnbq1rk1/ppppnpbp/4p1p1/6B1/3PP3/2P2N2/PP1Q1PP...,f6,2185,2162
1,r1b2rk1/4bppp/pqp2n2/3pN3/3P1B2/1PN2Q2/P1P2PPP...,Bd6,1876,1842
2,rn1qk1nr/pbp1bpp1/1p5p/3pP3/3P4/2NB1N2/PP3PPP/...,Qd7,1712,1609
3,r4rk1/1q2ppb1/1p1p2pp/2nP4/p2NP3/P1R1B1P1/1PQ2...,f3,1756,2151
4,rn2kbnr/ppq1pppp/2p5/5b2/3P4/2N2N2/PPP1BPPP/R1...,Be3,2394,2485


## Step 2: Improved Feature Engineering

The new encoding uses **12 binary planes** (one per piece type per color), each of size 8×8 (64 squares), flattened to a 768-element vector.

```
Plane 0: White pawns       Plane 6:  Black pawns
Plane 1: White knights     Plane 7:  Black knights
Plane 2: White bishops     Plane 8:  Black bishops
Plane 3: White rooks       Plane 9:  Black rooks
Plane 4: White queens      Plane 10: Black queens
Plane 5: White king        Plane 11: Black king
```

Plus 13 additional game-state features:
- Side to move (1)
- Castling rights: K, Q, k, q (4)
- En passant file, one-hot (8)

**Total: 781 features** (vs 65 in V1)

In [7]:
PIECE_TYPES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
COLORS = [chess.WHITE, chess.BLACK]

def fen_to_features(fen):
    """Convert FEN string to 781-element feature vector.
    
    Layout:
      [0:768]   12-plane binary board (piece type x color x square)
      [768]     side to move (1=white, 0=black)
      [769:773] castling rights (WK, WQ, BK, BQ)
      [773:781] en passant file, one-hot (files a-h)
    """
    board = chess.Board(fen)

    # 12 planes x 64 squares
    planes = np.zeros(768, dtype=np.float32)
    for color_idx, color in enumerate(COLORS):
        for piece_idx, piece_type in enumerate(PIECE_TYPES):
            plane = color_idx * 6 + piece_idx
            for sq in board.pieces(piece_type, color):
                planes[plane * 64 + sq] = 1.0

    # Side to move
    side = np.array([1.0 if board.turn == chess.WHITE else 0.0], dtype=np.float32)

    # Castling rights
    castling = np.array([
        float(board.has_kingside_castling_rights(chess.WHITE)),
        float(board.has_queenside_castling_rights(chess.WHITE)),
        float(board.has_kingside_castling_rights(chess.BLACK)),
        float(board.has_queenside_castling_rights(chess.BLACK)),
    ], dtype=np.float32)

    # En passant file (one-hot over 8 files)
    ep = np.zeros(8, dtype=np.float32)
    if board.ep_square is not None:
        ep[chess.square_file(board.ep_square)] = 1.0

    return np.concatenate([planes, side, castling, ep])


# Verify shape
sample = fen_to_features(positions_df.iloc[0]['fen'])
print(f'Feature vector shape: {sample.shape}')  # Expected: (781,)
print(f'  Board planes:      768')
print(f'  Side to move:        1')
print(f'  Castling rights:     4')
print(f'  En passant file:     8')
print(f'  Total:             781')

Feature vector shape: (781,)
  Board planes:      768
  Side to move:        1
  Castling rights:     4
  En passant file:     8
  Total:             781


## Step 3: Build Feature Matrix
Converting all 148K positions — this takes a moment.

In [9]:
print(f'Converting {len(positions_df):,} positions to feature vectors...')
print('(This may take ~60-90 seconds)')

X = np.array([fen_to_features(fen) for fen in positions_df['fen']])

# Encode move labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(positions_df['move_played'])
num_classes = len(label_encoder.classes_)

print(f'\nDone!')
print(f'  X shape:       {X.shape}')
print(f'  y shape:       {y.shape}')
print(f'  Unique moves:  {num_classes:,}')
print(f'  Memory usage:  {X.nbytes / 1e6:.1f} MB')

Converting 112,466 positions to feature vectors...
(This may take ~60-90 seconds)

Done!
  X shape:       (112466, 781)
  y shape:       (112466,)
  Unique moves:  1,841
  Memory usage:  351.3 MB


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train/Test split (80/20):')
print(f'  Training:   {len(X_train):,} samples')
print(f'  Test:       {len(X_test):,} samples')

Train/Test split (80/20):
  Training:   89,972 samples
  Test:       22,494 samples


## Step 4: Improved Model Architecture

Changes from V1:
- Input 781 features (vs 65)
- Wider layers: 1024 → 512 → 256 (vs 256 → 128 → 64)
- **BatchNormalization** after each Dense layer (stabilizes training, allows higher learning rate)
- **Early stopping** with `restore_best_weights=True` (no more overfitting after epoch 5)

In [12]:
model = keras.Sequential([
    layers.Input(shape=(781,)),

    layers.Dense(1024),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),

    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),

    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),

    layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1024)              800768    
                                                                 
 batch_normalization (BatchN  (None, 1024)             4096      
 ormalization)                                                   
                                                                 
 activation (Activation)     (None, 1024)              0         
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 512)               524800    
                                                                 
 batch_normalization_1 (Batc  (None, 512)              2048      
 hNormalization)                                        

## Step 5: Train with Early Stopping

In [14]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,              # Stop if no improvement for 5 epochs
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,              # Halve LR when plateau hit
    patience=3,
    min_lr=1e-6,
    verbose=1,
)

print('=' * 60)
print('TRAINING NEURAL NETWORK V2')
print('=' * 60)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

print('\nTraining complete!')
print(f'Stopped at epoch: {len(history.history["accuracy"])}')

TRAINING NEURAL NETWORK V2
Epoch 1/50


2026-03-01 11:13:48.535097: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


317/317 [==============================] - 4s 11ms/step - loss: 5.6712 - accuracy: 0.0669 - val_loss: 5.2706 - val_accuracy: 0.0975 - lr: 0.0010
Epoch 2/50
317/317 [==============================] - 4s 11ms/step - loss: 4.9331 - accuracy: 0.1162 - val_loss: 4.7303 - val_accuracy: 0.1463 - lr: 0.0010
Epoch 3/50
317/317 [==============================] - 4s 11ms/step - loss: 4.4003 - accuracy: 0.1628 - val_loss: 4.2366 - val_accuracy: 0.1902 - lr: 0.0010
Epoch 4/50
317/317 [==============================] - 4s 12ms/step - loss: 3.9121 - accuracy: 0.2086 - val_loss: 3.8753 - val_accuracy: 0.2362 - lr: 0.0010
Epoch 5/50
317/317 [==============================] - 4s 12ms/step - loss: 3.4893 - accuracy: 0.2585 - val_loss: 3.5540 - val_accuracy: 0.2838 - lr: 0.0010
Epoch 6/50
317/317 [==============================] - 4s 12ms/step - loss: 3.1430 - accuracy: 0.3075 - val_loss: 3.3090 - val_accuracy: 0.3182 - lr: 0.0010
Epoch 7/50
317/317 [==============================] - 4s 12ms/step - loss: 

KeyboardInterrupt: 

## Step 6: Evaluate — Top-1, Top-3, Top-5 Accuracy

Top-k accuracy tells us whether the correct move appears in the model's k most confident predictions. For a chess engine, top-3 and top-5 are practical: a search algorithm can evaluate the top candidates.

In [ ]:
print('Generating predictions on test set...')
y_pred_probs = model.predict(X_test, batch_size=256, verbose=0)

def topk_accuracy(y_true, y_pred_probs, k):
    """Fraction of samples where true label is in top-k predictions."""
    top_k_preds = np.argsort(y_pred_probs, axis=1)[:, -k:]
    correct = np.any(top_k_preds == y_true[:, None], axis=1)
    return correct.mean()

top1 = topk_accuracy(y_test, y_pred_probs, 1)
top3 = topk_accuracy(y_test, y_pred_probs, 3)
top5 = topk_accuracy(y_test, y_pred_probs, 5)

# V1 baseline for comparison
v1_accuracy = 0.0695

print('=' * 50)
print('MODEL V2 EVALUATION')
print('=' * 50)
print(f'  Top-1 accuracy:  {top1*100:.2f}%   (V1 baseline: {v1_accuracy*100:.2f}%)')
print(f'  Top-3 accuracy:  {top3*100:.2f}%')
print(f'  Top-5 accuracy:  {top5*100:.2f}%')
print(f'\n  Improvement (top-1): {(top1 - v1_accuracy)*100:+.2f}pp')
print(f'  Improvement factor:   {top1/v1_accuracy:.1f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

axes[0].plot(epochs_range, history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(epochs_range, history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].axhline(y=v1_accuracy, color='red', linestyle='--', label=f'V1 baseline ({v1_accuracy*100:.1f}%)', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy — V2 vs V1 Baseline')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['loss'], label='Train', linewidth=2)
axes[1].plot(epochs_range, history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Inference with Legal Move Masking

A key improvement for the chess application: at inference time we mask out illegal moves,
so the model only picks from moves that are actually legal in the given position.
This guarantees the engine never suggests an illegal move.

In [ ]:
def predict_move(fen, model, label_encoder, top_k=3):
    """Predict the best move for a position, restricted to legal moves.
    
    Returns a list of (move_san, probability) tuples, sorted by probability.
    Falls back to a random legal move if the model knows none of the legal moves.
    """
    board = chess.Board(fen)
    legal_moves_san = {board.san(m) for m in board.legal_moves}

    # Get raw probabilities from model
    features = fen_to_features(fen).reshape(1, -1)
    probs = model.predict(features, verbose=0)[0]

    # Build (move, prob) pairs for legal moves the encoder knows about
    known_classes = set(label_encoder.classes_)
    candidates = []
    for move_san in legal_moves_san:
        if move_san in known_classes:
            idx = label_encoder.transform([move_san])[0]
            candidates.append((move_san, float(probs[idx])))

    if not candidates:
        # Fallback: random legal move
        import random
        fallback = board.san(random.choice(list(board.legal_moves)))
        return [(fallback, 0.0)]

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]


# Test on a few positions from the test set
print('Sample predictions with legal move masking:\n')
sample_indices = np.random.choice(len(X_test), 5, replace=False)

for i, idx in enumerate(sample_indices):
    # Recover FEN from test set (same shuffle as train/test split)
    all_fens = positions_df['fen'].values
    all_moves = positions_df['move_played'].values
    _, test_fens, _, test_moves = train_test_split(
        all_fens, all_moves, test_size=0.2, random_state=42
    )
    fen = test_fens[idx]
    true_move = test_moves[idx]
    top_preds = predict_move(fen, model, label_encoder, top_k=3)

    predicted_move = top_preds[0][0]
    correct = '✓' if predicted_move == true_move else '✗'
    print(f'Position {i+1}: {correct}')
    print(f'  True move:  {true_move}')
    print(f'  Top 3:      {[(m, f"{p*100:.1f}%") for m, p in top_preds]}')
    print()

## Step 8: Save Model V2

In [ ]:
os.makedirs('../models', exist_ok=True)

model.save('../models/chess_move_predictor_v2.keras')

with open('../models/label_encoder_v2.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print('Model saved!')
print('  Model:         models/chess_move_predictor_v2.keras')
print('  Label encoder: models/label_encoder_v2.pkl')

print('\n' + '=' * 55)
print('SUMMARY')
print('=' * 55)
print(f'  Training samples:  {len(X_train):,}   (V1: 8,000)')
print(f'  Feature size:       781         (V1: 65)')
print(f'  Unique moves:       {num_classes:,}')
print(f'  Top-1 accuracy:    {top1*100:.2f}%       (V1: 6.95%)')
print(f'  Top-3 accuracy:    {top3*100:.2f}%')
print(f'  Top-5 accuracy:    {top5*100:.2f}%')